In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime
import re
import ast
import json
import os
from tqdm.notebook import tqdm  # or from tqdm import tqdm


# Translation Table

In [ ]:
# datasets from Translation_02_Drug_Disease

In [ ]:
# for the old analysis translation_table_drug_disease_OLD.csv
translation_table_full = pd.read_csv("/shares/animalwelfare.crs.uzh/Preclinical_Pipeline/10_use_case_disease_focus/out/translation_table_drug_disease_FULL.csv") # from Translation_02_Drug_Disease
translation_table_full.shape

In [ ]:
fda_only_translated = pd.read_csv("/shares/animalwelfare.crs.uzh/Preclinical_Pipeline/10_use_case_disease_focus/out/translation_table_drug_disease_neuro_FDA_only.csv")
fda_only_translated['fda_only']=True
fda_only_translated.shape

In [ ]:
translation_table = pd.concat([translation_table_full, fda_only_translated], ignore_index=True)

In [ ]:
n_diseases = translation_table["merged_mondo_label"].nunique()
n_drugs = translation_table["merged_umls_label"].nunique()

print("\n=== Entity coverage summary ===")
print(f"Translation table shape: {translation_table.shape,}")
print(f"Total unique diseases: {n_diseases:,}")
print(f"Total unique drugs: {n_drugs:,}")

In [ ]:
translation_table.head()

# Filter for neuro pairs

### neuro filter 17

In [ ]:
translation_table = pd.concat([translation_table_full, fda_only_translated], ignore_index=True)

In [ ]:
disease_keywords = {
    "multiple_sclerosis": ["multiple sclerosis"],
    "alzheimers_disease": ["alzheimer"],
    "parkinsons_disease": ["parkinson"],
    "huntingtons_disease": ["huntington"],
    "motor_neuron_disease": ["motor neuron", "amyotrophic lateral sclerosis", "als"],
    "neuropathic_pain": ["neuropathic pain", "neuralgia", "neuropathy"],
    "prion_disease": ["prion", "creutzfeldt", "cjd"],
    "spinal_cord_injury": ["spinal cord injury"],
    "traumatic_brain_injury": ["traumatic brain injury", "tbi"],
    "subarachnoid_hemorrhage": ["subarachnoid hemorrhage"],
    "stroke": ["stroke", "cerebral infarction", "brain ischem"],
    "schizophrenia": ["schizophrenia", "psychosis"],
    "depression": ["depression", "major depressive"],
    "addiction": ["addiction", "substance use", "dependence"],
    "autism": ["autism", "autism spectrum"],
    "brain_tumor": ["brain tumor", "glioblastoma", "glioma", "meningioma"],
    "epilepsy": ["epilepsy", "seizure"]
}

df = translation_table.copy()
df["disease_part"] = (
    df["normalized_key"]
    .str.lower()
    .str.split("<>")
    .str[0]
    .str.strip()
)

def match_disease(text):
    for label, keywords in disease_keywords.items():
        for kw in keywords:
            if kw in text:
                return label
    return None

df["matched_disease"] = df["disease_part"].apply(match_disease)

translation_table = df[df["matched_disease"].notna()]

n_diseases = translation_table["merged_mondo_label"].nunique()
n_drugs = translation_table["merged_umls_label"].nunique()

print("\n=== Entity coverage summary ===")
print(f"Translation table shape: {translation_table.shape,}")
print(f"Total unique diseases: {n_diseases:,}")
print(f"Total unique drugs: {n_drugs:,}")

In [ ]:
translation_table.to_csv("/shares/animalwelfare.crs.uzh/Preclinical_Pipeline/10_use_case_disease_focus/out/translation_table_drug_disease_NEURO_17.csv", index=False)

### neuro filter library

In [ ]:
translation_table = pd.concat([translation_table_full, fda_only_translated], ignore_index=True)

In [ ]:
neuro = pd.read_csv("./data/list_neuro_conditions.csv")
disease_terms = (
    neuro["disease_mondo_term_norm"]
    .dropna()
    .astype(str)
    .str.strip()
    .str.lower()
)

terms_set = set(disease_terms)
terms_list = list(terms_set)
terms = sorted({
    t.strip().lower()
    for t in terms_list
    if isinstance(t, str) and t.strip()
}, key=len, reverse=True)

terms = [t for t in terms if len(t) > 4]
pattern = re.compile("|".join(map(re.escape, terms)))

s = (
    translation_table["normalized_key"]
    .fillna("")
    .astype(str)
    .str.lower()
)

terms_l = [t.lower() for t in terms]  # ensure terms are lowercase too

matched = np.full(len(s), None, dtype=object)
mask = np.zeros(len(s), dtype=bool)

chunk_size = 500

for i in tqdm(range(0, len(terms_l), chunk_size)):
    chunk_terms = terms_l[i : i + chunk_size]

    # Build alternation regex for this chunk
    pattern_str = "|".join(map(re.escape, chunk_terms))

    # Extract first match from this chunk (per row); NaN if none
    m = s.str.extract(f"({pattern_str})", expand=False)

    hit = m.notna().to_numpy()

    # update overall mask
    mask |= hit

    # only fill matched term where we don't already have one
    fill_idx = hit & (matched == None)
    matched[fill_idx] = m.to_numpy()[fill_idx]

translation_table["matched_term"] = matched
translation_table = translation_table[mask]

translation_table.shape


In [ ]:
n_diseases = translation_table["merged_mondo_label"].nunique()
n_drugs = translation_table["merged_umls_label"].nunique()

print("\n=== Entity coverage summary ===")
print(f"Translation table shape: {translation_table.shape,}")
print(f"Total unique diseases: {n_diseases:,}")
print(f"Total unique drugs: {n_drugs:,}")

In [ ]:
translation_table.normalized_key.nunique()

In [ ]:
translation_table.to_csv("/shares/animalwelfare.crs.uzh/Preclinical_Pipeline/10_use_case_disease_focus/out/translation_table_drug_disease_NEURO.csv", index=False)